# **Modern Vision-Language Image Captioning (VLM Architecture)**

### **Why this replaces the 2017 toy approach:**
1. **No Single Vector Bottleneck:** Instead of squashing pixels into 1 single vector, high-resolution **spatial patch grids** (576+ tokens) preserve exact spatial coordinates.
2. **Pre-Trained Multimodal Language Model:** The language model already knows English grammar, physics, and world context (no training from scratch).
3. **Zero-Shot & Fine-Tuning Ready:** Instant human-level captions with Beam Search, plus optional domain adaptation.

### **1. Setup & Environment**

In [1]:
!pip install -q transformers torch torchvision pillow requests

import os
import io
import json
import requests
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch as tr
from torch.utils.data import Dataset, DataLoader
from transformers import BlipProcessor, BlipForConditionalGeneration

# Verify GPU acceleration
device = tr.device('cuda' if tr.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU Name: {tr.cuda.get_device_name(0)}")

Using device: cuda
GPU Name: Tesla T4


### **2. Loading State-of-the-Art Industry Model (Salesforce BLIP-Large)**

In [2]:
# Salesforce BLIP is pre-trained on 129M image-text pairs with spatial visual tokens
model_id = "Salesforce/blip-image-captioning-large"
print(f"Loading {model_id} onto {device}...")

processor = BlipProcessor.from_pretrained(model_id)
model = BlipForConditionalGeneration.from_pretrained(model_id).to(device)
model.eval()

print("Model loaded successfully!")

Loading Salesforce/blip-image-captioning-large onto cuda...


preprocessor_config.json:   0%|          | 0.00/445 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.60k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/527 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.88GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/616 [00:00<?, ?it/s]

Model loaded successfully!


### **3. Production Inference Function (Beam Search & Nucleus Sampling [also called TOP-P sampling])**

## 1. Notebook Architecture & Core Upgrades

`5_2.ipynb` upgrades the workflow from a scratch toy model (`5_1.ipynb`) to a production-grade **Vision-Language Model (Salesforce BLIP-Large)**:

| Feature / Component | Scratch Model (`5_1.ipynb`) | Advanced VLM (`5_2.ipynb`) |
| :--- | :--- | :--- |
| **Visual Feature Extraction** | Single global pooled vector ($1 \times 512$). All spatial layout (left, right, depth, background) is squashed into 1 point. | **Spatial Visual Patch Grid** ($576+$ patch tokens), preserving fine-grained 2D coordinates of objects across the image. |
| **Language Decoder** | Transformer Decoder trained from scratch on limited MS-COCO data. | **Pre-trained Multimodal Transformer (BLIP)** pre-trained on 129M image-text pairs with built-in English grammar and world knowledge. |
| **Decoding Strategy** | Greedy `argmax` (prone to repetition and local-optimum dead ends). | **Beam Search (`num_beams=5`)** with repetition penalty and length normalization. |
| **Training Pipeline** | Static array feature caching with high RAM footprint. | PyTorch `Dataset` with **dynamic on-the-fly tokenization** and Mixed Precision (`torch.cuda.amp.autocast`) fine-tuning. |

## 2. The Text Generation Dilemma (Decoding Problem)

At each step $t$, the decoder outputs a probability distribution over the entire vocabulary $V$ (e.g., 30,000 words):

$$P(w_t \mid w_1, w_2, \dots, w_{t-1}, \text{Image})$$

How do we decide which token to select next?

* **Greedy Search (`argmax`):** Always picks the single highest-probability token at each step.
  * *Flaw:* A word with the highest immediate probability can lead into low-probability, grammatically broken sentences (*"A dog dog dog on the..."*).
* **Exhaustive Search:** Tests every possible sequence combination.
  * *Flaw:* Combinatorial explosion makes this computationally impossible ($|V|^T = 30000^{20} \approx 10^{89}$ possibilities).

To resolve this, modern decoders use **Beam Search** (deterministic optimization) or **Nucleus Sampling** (stochastic controlled sampling).

## 3. What is Beam Search?

**Beam Search** is a heuristic search algorithm that maintains a set of the top-$B$ most probable candidate sequences (where $B$ is the **Beam Width**, e.g., $B=5$).

### Step-by-Step Mechanics:

1. **Step 1:** The model generates probability scores for all vocabulary words given the image. Keep only the top $B$ words.
2. **Step $t$:** For each of the $B$ active candidates, generate the next-token probability distribution over the vocabulary ($B \times |V|$ candidate extensions).
3. **Cumulative Scoring:** Score each sequence candidate using joint log-probabilities:
   $$\text{Score}(w_1, \dots, w_t) = \sum_{i=1}^t \log P(w_i \mid w_{<i})$$
4. **Pruning:** Rank all $B \times |V|$ candidates and keep **only the top $B$ highest-scoring sequences**. Discard all others.
5. **Termination:** When a hypothesis outputs the end-of-sequence token (`<end>` / `[SEP]`), it is saved. The search stops once $B$ sequences finish or maximum length is reached.

```
Step 1 (Keep Top-2):
  ├── "A"       (P = 0.60)
  └── "The"     (P = 0.30)

Step 2 (Expand all combinations, prune back to Top-2):
  ├── "A dog"      (0.60 × 0.70 = 0.42)  <-- KEPT
  ├── "A cat"      (0.60 × 0.20 = 0.12)
  ├── "The woman"  (0.30 × 0.80 = 0.24)  <-- KEPT
  └── "The car"    (0.30 × 0.10 = 0.03)

Step 3 (Expand only the 2 winners):
  ...
```

### Key Parameters in `5_2.ipynb`:
* `num_beams=5`: Maintains 5 distinct hypotheses at each generation step.
* `repetition_penalty=1.2`: Penalizes logits of previously generated words to prevent repetitive cycles.
* `length_penalty=1.0`: Since multiplying probabilities $\le 1$ naturally penalizes longer captions, length normalization balances scores:
  $$\text{Normalized Score} = \frac{\sum \log P}{\text{Length}^{\alpha}}$$

**Best for:** Factual accuracy, deterministic benchmarks (BLEU, CIDEr), catalog/scientific descriptions.

## 4. What is Nucleus Sampling (Top-$p$ Sampling)?

While Beam Search optimizes for maximum probability, human language does not always use the most statistically probable word. Beam search can sound repetitive or generic (*"A woman standing in a room"*).

Sampling introduces natural variation, but:
* **Pure Random Sampling:** Often picks low-probability "hallucination" words from the tail of the distribution.
* **Top-$k$ Sampling:** Restricts choices to a fixed number $k$ (e.g., $k=10$). However, if the distribution is sharp (one obvious word with 98% probability), the other 9 are bad choices. If the distribution is flat (50 valid synonyms), $k=10$ cuts off 40 plausible options.

**Nucleus (Top-$p$) Sampling** dynamically determines the candidate pool size by selecting the smallest set of words whose cumulative probability exceeds threshold $p$ (e.g., $p = 0.90$):

$$\sum_{i \in \text{Nucleus}} P(w_i) \ge p$$

```
Ranked Vocabulary Probabilities:
"dog":     0.60 ──┐
"puppy":   0.25 ──┼── Cumulative = 0.85 (under 0.90, include next)
"canine":  0.08 ──┴── Cumulative = 0.93 (>= 0.90! Cut off here)
-------------------- Nucleus Cutoff --------------------
"table":   0.03   (Discarded)
"car":     0.02   (Discarded)
"sky":     0.02   (Discarded)
```

### The Role of Temperature ($\tau$):
Before computing the softmax probabilities, logits $z_i$ are scaled by temperature $\tau$:

$$P(w_i) = \frac{\exp(z_i / \tau)}{\sum_j \exp(z_j / \tau)}$$

* **Low Temperature ($\tau < 0.5$):** Sharpens the distribution. The top choices dominate (behaves closer to greedy/beam search).
* **High Temperature ($\tau > 0.8$):** Flattens the distribution. Words have closer probabilities, increasing diversity and creativity.

**Best for:** Creative writing, social media captions, generating multiple varied descriptions for the same image.

## 5. Comparison & Code Implementation

| Characteristic | Beam Search | Nucleus Sampling (Top-$p$) |
| :--- | :--- | :--- |
| **Output Type** | Deterministic (same input = same output) | Stochastic (different output each run) |
| **Vocabulary Diversity** | Conservative, standard vocabulary | Rich, expressive, varied phrasing |
| **Failure Mode** | Generic or repetitive phrasing | Minor risk of hallucination if $p$ or $\tau$ is too high |
| **Compute Cost** | Higher ($B\times$ passes per token) | Lower ($1\times$ pass per token) |

### Switching Between Strategies in `5_2.ipynb`:

```python
# Strategy 1: Beam Search (High Accuracy / Benchmark Evaluation)
beam_output = model.generate(
    **inputs,
    num_beams=5,
    max_new_tokens=30,
    min_length=8,
    repetition_penalty=1.2,
    length_penalty=1.0
)
caption_beam = processor.decode(beam_output[0], skip_special_tokens=True).strip()

# Strategy 2: Nucleus Sampling (Diverse / Creative Phrasing)
sampling_output = model.generate(
    **inputs,
    do_sample=True,          # Enables sampling mode
    top_p=0.90,              # Cumulative probability threshold (90%)
    temperature=0.7,         # Softens logit distribution
    max_new_tokens=30,
    min_length=8,
    repetition_penalty=1.2
)
caption_sampled = processor.decode(sampling_output[0], skip_special_tokens=True).strip()
```

### Beam Search: Pros and Cons

#### Pros
* **Deterministic and Reproducible:** Always produces the exact same caption for the same image. Essential for production systems, testing, and debugging.
* **Factual and Grammatically Accurate:** Optimizes for the highest joint probability, reducing hallucinations and keeping captions strictly grounded in image features.
* **Higher Benchmark Scores:** Consistently achieves top scores on standard evaluation metrics like BLEU, ROUGE, and CIDEr by matching reference ground truths.

#### Cons
* **Generic and Bland Phrasing:** Consistently picks the safest words, producing dry captions (e.g., "a dog on the grass" instead of describing the breed or action).
* **Susceptible to Repetition Loops:** Long sequences often get trapped repeating identical phrases.
* **Higher Compute and Memory Overhead:** A beam width of B (e.g., B=5) requires 5 parallel decoder computations at each step, making inference slower.

### Nucleus (Top-p) Sampling: Pros and Cons

#### Pros
* **Natural and Expressive Language:** Mimics human communication with varied sentence structure and richer vocabulary.
* **Adaptive Candidate Selection:** Unlike fixed Top-k, Top-p dynamically expands or shrinks the candidate word pool depending on the model's confidence.
* **Fast and Lightweight:** Runs only a single sequence path (B=1), requiring significantly less GPU memory and generating tokens faster.
* **Eliminates Repetitive Loops:** Probabilistic sampling naturally prevents deterministic looping traps.

#### Cons
* **Non-Deterministic:** Produces a different caption on each run, making strict output verification difficult.
* **Risk of Minor Hallucinations:** Rolling a random dice means improbable or slightly inaccurate descriptor words can occasionally be selected.
* **Requires Careful Tuning:** Needs both `top_p` and `temperature` balanced properly; high values produce nonsensical output while low values collapse into greedy search.
* **Lower Automated Metric Scores:** Creative synonyms reduce direct word-overlap with standard benchmark references.

### When to Use What

#### Use Beam Search For:
* **Medical, Technical, and Legal Tasks:** Diagnostic imaging, X-ray report generation, and document processing where zero error is tolerated.
* **Automated Benchmarking:** Computing standard CIDEr/BLEU scores on MS-COCO or Flickr8k test splits.
* **E-Commerce and Cataloging:** Generating concise, uniform product descriptions across thousands of inventory items.
* **Translation and Summarization:** Tasks where the objective is strict preservation of meaning rather than stylistic flair.

#### Use Nucleus Sampling For:
* **Interactive AI Assistants and Chatbots:** Conversational models (ChatGPT, Claude) requiring engaging, non-robotic dialogue.
* **Social Media and Marketing Copy:** Creating varied captions, ad headlines, and descriptions with personality.
* **Candidate Variation:** Generating 3 to 5 diverse caption options for a user to choose from.
* **Creative and Long-Form Generation:** Storytelling, open-ended writing, and brainstorming.

In [1]:
def predict_caption(
    image_source,
    prompt=None,
    num_beams=5,
    max_new_tokens=30,
    min_length=8,
    repetition_penalty=1.2,
    display_img=True
):
    """
    Generates high-precision captions for local images, PIL images, or URLs. Supports unconditional captioning and prompt-guided captioning.
    """
    # 1. Loading image from URL, file path, or PIL object
    if isinstance(image_source, str):
        if image_source.startswith("http://") or image_source.startswith("https://"):
            headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
            resp = requests.get(image_source, headers=headers)
            img = Image.open(io.BytesIO(resp.content)).convert('RGB')
        else:
            img = Image.open(image_source).convert('RGB')
    else:
        img = image_source.convert('RGB')

    # 2. Preprocessing images with multimodal spatial tokens
    if prompt:
        inputs = processor(img, text=prompt, return_tensors="pt").to(device)
    else:
        inputs = processor(img, return_tensors="pt").to(device)

    # 3. Generating with Beam Search
    with tr.no_grad():
        output = model.generate(
            **inputs,
            num_beams=num_beams,
            max_new_tokens=max_new_tokens,
            min_length=min_length,
            repetition_penalty=repetition_penalty,
            length_penalty=1.0
        )

    caption = processor.decode(output[0], skip_special_tokens=True).strip()

    # 4. Display result
    if display_img:
        plt.figure(figsize=(6, 6))
        plt.imshow(img)
        plt.title(f"Generated Caption:\n{caption}", fontsize=11)
        plt.axis('off')
        plt.show()

    return caption

### **4. Benchmark Against Previous Hard Cases**

Let's test the exact two images that confused the toy model (the red cardigan on the rocky shore and the blonde woman with sun flare):

In [ ]:
# Test 1: The woman with red cardigan and city skyline
url_1 = "https://images.unsplash.com/photo-1791514122824-a10c01b1864e?q=80&w=687&auto=format&fit=crop&ixlib=rb-4.1.0&ixid=NHwxMjA3fDA%3D"
print("Generating for Test Image 1...")
cap_1 = predict_caption(url_1)
print(f"Caption 1: {cap_1}")

In [ ]:
# Test 2: The blonde woman in sunny field (previously hallucinated 'red umbrella' due to sun flare)
url_2 = "https://lh3.googleusercontent.com/hwau7OVWx96XaME5KpRuJ0I_MscrerK6SbRH7z3h"
print("Generating for Test Image 2...")
cap_2 = predict_caption(url_2)
print(f"Caption 2: {cap_2}")

### **5. Fine-Tuning the VLM on Custom / MS-COCO Data (The Industry Recipe)**

In industry, if you need domain adaptation (e.g. medical, e-commerce, sports), you fine-tune the multimodal decoder while keeping the vision encoder frozen:

In [ ]:
# PyTorch Dataset with on-the-fly tokenization (Zero RAM bottleneck)
class CaptionDataset(Dataset):
    def __init__(self, df, processor, max_length=32):
        self.df = df
        self.processor = processor
        self.max_length = max_length
        
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
        item = self.df.iloc[idx]
        image = Image.open(item['image_path']).convert('RGB')
        caption = item['caption']
        
        # Tokenize text and normalize image pixels together
        encoding = self.processor(
            images=image,
            text=caption,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )
        
        # Remove batch dim added by processor
        encoding = {k: v.squeeze(0) for k, v in encoding.items()}
        encoding['labels'] = encoding['input_ids'].clone()
        return encoding

print("CaptionDataset class defined successfully.")

In [ ]:
def fine_tune_blip(train_df, val_df, epochs=2, batch_size=8, lr=5e-6):
    """
    Fine-tunes BLIP using low learning rate (5e-6) with AdamW.
    """
    train_dataset = CaptionDataset(train_df, processor)
    val_dataset = CaptionDataset(val_df, processor)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    
    optimizer = tr.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    scaler = tr.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        
        for step, batch in enumerate(train_loader):
            pixel_values = batch['pixel_values'].to(device)
            input_ids = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)
            
            optimizer.zero_grad()
            with tr.cuda.amp.autocast(enabled=(device.type == 'cuda')):
                outputs = model(input_ids=input_ids, pixel_values=pixel_values, labels=labels)
                loss = outputs.loss
                
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            total_loss += loss.item()
            if (step + 1) % 50 == 0:
                print(f"Epoch [{epoch+1}/{epochs}] Step [{step+1}/{len(train_loader)}] Loss: {loss.item():.4f}")
                
        avg_train_loss = total_loss / len(train_loader)
        print(f"--> Epoch {epoch+1} Finished. Average Loss: {avg_train_loss:.4f}")
        
    print("Fine-tuning complete!")

### **6. Save & Export Model Weights**

In [ ]:
save_dir = "./blip_custom_captioner"
os.makedirs(save_dir, exist_ok=True)

model.save_pretrained(save_dir)
processor.save_pretrained(save_dir)
print(f"Model and processor successfully saved to: {save_dir}")